[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/karzit/temp/blob/master/notebooks/ml-curriculum/00_python_essentials/00_python_essentials.ipynb)

# 00. NumPy / Pandas / PyTorch 필수 라이브러리 실습

> 이 커리큘럼 전체(02~06)에서 계속 쓰이는 세 라이브러리를 먼저 손에 익힙니다. ML/DL 이론 자체보다
> "행렬 연산, 표 형태 데이터 다루기, 텐서 기본기"에 집중하는 준비 단계입니다.

## 왜 필요한가
- **NumPy**: 벡터/행렬 연산의 기본. `02_linear_regression`에서 Gradient Descent를 직접 구현할 때 사용합니다.
- **Pandas**: CSV 등 표 형태 데이터를 불러오고 가공하는 표준 도구. `01_basic_classification`부터 계속 등장합니다.
- **PyTorch**: `04_neural_networks`부터 신경망을 만들 때 쓰는 딥러닝 프레임워크. 텐서(Tensor) 연산과
  자동 미분(autograd)이 핵심입니다.

이미 세 라이브러리에 익숙하다면 이 노트북은 건너뛰고 바로 [01_basic_classification](../01_basic_classification/01_basic_classification.ipynb)로 가도 됩니다.


## 실습 0. Colab 환경 설정

In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules
print("Running in Colab:", IN_COLAB)
if IN_COLAB:
    !pip install -q numpy pandas torch


## 실습 1. NumPy — 배열과 행렬 연산

파이썬 `list`와 달리 `ndarray`는 같은 타입의 원소를 연속된 메모리에 저장해서 빠르고,
원소별 연산(브로드캐스팅)을 반복문 없이 처리할 수 있습니다.

In [ ]:
import numpy as np

a = np.array([1, 2, 3])
b = np.array([[1, 2, 3], [4, 5, 6]])

print("a:", a, "shape:", a.shape, "dtype:", a.dtype)
print("b:\n", b, "shape:", b.shape)


### 1-1. 생성 함수와 인덱싱/슬라이싱

In [ ]:
zeros = np.zeros((2, 3))
ones = np.ones((2, 3))
rng = np.random.default_rng(42)
rand = rng.normal(size=(2, 3))
arange = np.arange(0, 10, 2)
linspace = np.linspace(0, 1, 5)

print(zeros)
print(ones)
print(rand)
print(arange)
print(linspace)


In [ ]:
m = np.arange(12).reshape(3, 4)
print(m)
print("첫 행:", m[0])
print("첫 열:", m[:, 0])
print("2x2 부분:\n", m[:2, :2])
print("짝수만 (불리언 인덱싱):", m[m % 2 == 0])


### 1-2. 브로드캐스팅과 벡터화 연산

파이썬 반복문 대신 배열 전체에 연산을 한 번에 적용합니다. `02_linear_regression`에서
`W * x + b`처럼 스칼라와 배열을 바로 곱하고 더할 수 있는 이유가 브로드캐스팅입니다.

In [ ]:
x = np.array([1.0, 2.0, 3.0, 4.0])
W, b = 2.0, 3.0

y_pred = W * x + b          # 반복문 없이 전체 원소에 적용 (브로드캐스팅)
print("y_pred:", y_pred)

# 벡터화 vs 반복문 속도 비교
big = rng.normal(size=1_000_000)

import time
start = time.perf_counter()
total = sum(v * 2 for v in big)         # 순수 파이썬 반복문
py_time = time.perf_counter() - start

start = time.perf_counter()
total_np = (big * 2).sum()              # NumPy 벡터화
np_time = time.perf_counter() - start

print(f"python loop: {py_time:.4f}s, numpy: {np_time:.4f}s")


### 1-3. 행렬 연산 — Linear Regression의 핵심 연산

In [ ]:
X = np.array([[1, 2], [3, 4], [5, 6]])   # (3, 2) — 샘플 3개, 특성 2개
W = np.array([[0.5], [1.5]])              # (2, 1) — 가중치

H = X @ W                                  # 행렬곱 (내적) — H(X) = XW
print("X @ W:\n", H)

print("전치:\n", X.T)
print("합/평균/최대:", X.sum(), X.mean(), X.max())
print("축 지정 합(행 방향, axis=1):", X.sum(axis=1))


### NumPy 연습 문제
1. `(5, 5)` 크기의 항등 행렬을 만들고, 모든 원소에 3을 곱한 뒤 행 단위 합을 구해보세요. (`np.eye` 참고)
2. `rng.integers(0, 100, size=20)`로 배열을 만들고, 50 이상인 값만 골라 평균을 구해보세요.
3. `(4, 3)` 행렬과 `(3, 2)` 행렬을 만들어 행렬곱을 해보고, 순서를 바꾸면 왜 오류가 나는지 `shape`로 설명해보세요.

## 실습 2. Pandas — 표 형태 데이터 다루기

`DataFrame`은 행/열에 이름이 붙은 표입니다. NumPy 배열이 "숫자 행렬"이라면, Pandas는
"엑셀 시트"에 가깝습니다. CSV를 읽고, 필터링하고, 그룹별로 집계하는 게 핵심 작업입니다.

In [ ]:
import pandas as pd

data = {
    "name": ["Alice", "Bob", "Carol", "Dave", "Eve"],
    "score": [88, 92, 79, 65, 95],
    "team": ["A", "B", "A", "B", "A"],
}
df = pd.DataFrame(data)
df


### 2-1. 탐색 — head/info/describe

In [ ]:
print(df.head(3))
print(df.info())
print(df.describe())


### 2-2. 선택과 필터링

In [ ]:
print(df["score"])                    # 열 선택 -> Series
print(df[["name", "score"]])           # 여러 열 선택 -> DataFrame
print(df[df["score"] >= 80])           # 조건 필터링
print(df.loc[df["team"] == "A", "name"])  # 라벨 기반 선택


### 2-3. 새 열 추가와 정렬

In [ ]:
df["passed"] = df["score"] >= 70
df_sorted = df.sort_values("score", ascending=False)
print(df)
print(df_sorted)


### 2-4. 그룹별 집계 — groupby

In [ ]:
print(df.groupby("team")["score"].mean())
print(df.groupby("team").agg(avg_score=("score", "mean"), count=("name", "count")))


### 2-5. CSV 읽고 쓰기

`01_basic_classification`, `02_linear_regression`에서 계속 쓰는 패턴입니다.

In [ ]:
import os

os.makedirs("../../../data", exist_ok=True)
df.to_csv("../../../data/sample_scores.csv", index=False)

loaded = pd.read_csv("../../../data/sample_scores.csv")
loaded


### Pandas 연습 문제
1. `df`에 `"age"` 열을 임의로 추가하고, 나이 순으로 정렬해보세요.
2. `team`별로 `passed`가 `True`인 인원 수를 구해보세요. (`groupby` + `sum` 힌트)
3. `score`가 팀 평균보다 높은 사람만 골라내보세요. (`groupby().transform()` 또는 팀별 평균을 먼저 구해 병합)

## 실습 3. PyTorch — 텐서와 자동 미분(autograd)

PyTorch의 `Tensor`는 NumPy `ndarray`와 거의 같은 인터페이스를 가지면서, GPU 연산과
**자동 미분**을 지원합니다. `04_neural_networks`부터 이 자동 미분으로 Gradient Descent를
직접 계산하지 않고도 backpropagation을 수행합니다.

In [ ]:
import torch

t1 = torch.tensor([1.0, 2.0, 3.0])
t2 = torch.zeros(2, 3)
t3 = torch.from_numpy(np.array([[1, 2], [3, 4]], dtype=np.float32))

print(t1, t1.shape, t1.dtype)
print(t2)
print(t3)
print("numpy로 변환:", t3.numpy())


### 3-1. 텐서 연산 (NumPy와 거의 동일)

In [ ]:
a = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
b = torch.tensor([[5.0, 6.0], [7.0, 8.0]])

print("elementwise:\n", a * b)
print("행렬곱:\n", a @ b)
print("전치:\n", a.T)
print("reshape (view):\n", a.view(4))


### 3-2. 자동 미분 — `requires_grad`와 `.backward()`

`02_linear_regression`에서 numpy로 손수 미분(`dW`, `db`)을 계산했던 것을, PyTorch는
`.backward()` 호출 한 번으로 자동 계산해줍니다.

In [ ]:
W = torch.tensor(0.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)

x = torch.tensor([1.0, 2.0, 3.0, 4.0])
y = torch.tensor([5.0, 7.0, 9.0, 11.0])   # y = 2x + 3

y_pred = W * x + b
cost = ((y_pred - y) ** 2).mean()

cost.backward()      # dCost/dW, dCost/db를 자동 계산

print("W.grad:", W.grad.item())
print("b.grad:", b.grad.item())


### 3-3. Gradient Descent 루프 — numpy 버전과 비교

In [ ]:
W = torch.tensor(0.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)
lr = 0.01

for epoch in range(200):
    y_pred = W * x + b
    cost = ((y_pred - y) ** 2).mean()

    cost.backward()
    with torch.no_grad():          # 파라미터 업데이트는 미분 추적 대상에서 제외
        W -= lr * W.grad
        b -= lr * b.grad
        W.grad.zero_()             # gradient는 누적되므로 매 epoch마다 초기화
        b.grad.zero_()

    if epoch % 40 == 0:
        print(f"epoch {epoch:3d}  cost={cost.item():.4f}  W={W.item():.3f}  b={b.item():.3f}")

print(f"최종: W={W.item():.3f}, b={b.item():.3f} (목표: W=2, b=3)")


### PyTorch 연습 문제
1. `requires_grad=True`인 스칼라 텐서 `x = torch.tensor(3.0, requires_grad=True)`에 대해
   `y = x ** 2 + 2 * x`의 미분값을 `.backward()`로 구하고, 손으로 계산한 `dy/dx = 2x + 2`와 비교해보세요.
2. 위 3-3의 학습 루프에서 `W.grad.zero_()`를 지우면 어떤 일이 벌어지는지 실행해서 확인해보세요.
3. `torch.tensor`와 `np.array` 사이를 `.numpy()` / `torch.from_numpy()`로 왕복 변환해보세요.

## 정리
- **NumPy**: 배열/행렬 연산, 브로드캐스팅 — 딥러닝 프레임워크 없이 수식을 직접 구현할 때 기본기가 됩니다.
- **Pandas**: 표 형태 데이터를 불러오고(`read_csv`), 필터링/집계(`groupby`)하는 표준 워크플로우.
- **PyTorch**: `Tensor` + `autograd`로 신경망의 gradient를 자동 계산합니다. `02_linear_regression`에서
  numpy로 손수 구현했던 미분을 `04_neural_networks`부터는 PyTorch가 대신 해줍니다.

다음 노트북: [01_basic_classification](../01_basic_classification/01_basic_classification.ipynb)에서
scikit-learn 파이프라인 전체 흐름을 감 잡아봅니다.

**해설/정답**: [00_python_essentials_solutions.ipynb](00_python_essentials_solutions.ipynb)
